# 🚚 DeliverIQ – Delivery Time Prediction
**End-to-end ML pipeline**: data cleaning → feature engineering → model training → evaluation → feature importance → deployment-ready export.

## 🔹 Step 1: Load Dataset

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('Food_Delivery_Times.csv')
print(f'Shape: {df.shape}')
df.head()

## 🔹 Step 2: Basic Inspection

In [ ]:
print('=== Data Types & Non-Null Counts ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Basic Statistics ===')
df.describe()

## 🔹 Step 3: Data Cleaning
- Drop `Order_ID` (identifier, not a predictor)
- Impute missing **categoricals** with **mode**
- Impute missing **numerics** with **median**

In [ ]:
# Drop non-feature column
df.drop(columns=['Order_ID'], inplace=True)

# Impute categorical columns with mode
cat_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Impute numeric column with median
df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].median(), inplace=True)

print('Missing values after cleaning:', df.isnull().sum().sum())
df.head()

## 🔹 Step 4: Features & Target

In [ ]:
X = df.drop('Delivery_Time_min', axis=1)
y = df['Delivery_Time_min']
print(f'Features: {X.shape[1]} columns, {X.shape[0]} rows')
print(f'Target range: {y.min()} – {y.max()} minutes')

## 🔹 Step 5: Encode Categorical Columns

In [ ]:
X = pd.get_dummies(X, drop_first=True)
print(f'Encoded features: {X.shape[1]}')
print(list(X.columns))

## 🔹 Step 6: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]} samples')

## 🔹 Step 7: Train Model – Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print('✅ Model trained successfully!')

## 🔹 Step 8: Evaluate Model

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
r2  = r2_score(y_test, preds)

print(f'Mean Absolute Error (MAE) : {mae:.2f} minutes')
print(f'R² Score                  : {r2:.4f}')

## 🔹 Step 9: Feature Importance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

importances = model.feature_importances_
features    = X.columns

# Sort by importance
sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(features)))
bars = ax.barh(features[sorted_idx], importances[sorted_idx], color=colors)
ax.set_title('Feature Importance – RandomForestRegressor', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.bar_label(bars, fmt='%.3f', padding=4, fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as feature_importance.png')

## 🔹 Step 10: Save Model Artefacts

In [ ]:
import pickle

# Save trained model
pickle.dump(model, open('model.pkl', 'wb'))

# Save feature column names (needed for web app alignment)
feature_columns = list(X.columns)
pickle.dump(feature_columns, open('feature_columns.pkl', 'wb'))

print('✅ Saved: model.pkl')
print('✅ Saved: feature_columns.pkl')
print(f'   → {len(feature_columns)} feature columns stored')